# Gold — Modelagem territorial por município e ano



Construção da camada Gold do ENEM para a UF configurada (1998–2025).



## Princípios



- O Gold físico preserva o nível dos registros.

- A referência territorial é município × ano.

- Não há associação individual entre `PARTICIPANTES` e `RESULTADOS`.

- 1998–2000: território de residência.

- 2001–2007: território da escola.

- 2008–2025: território da prova.

- 2024–2025 utilizam `PARTICIPANTES` e `RESULTADOS` como fontes complementares.

- Bronze, Staging e Silver não são modificados.

- A auditoria do Gold é realizada em notebook separado.


Fontes Silver são descobertas diretamente pela estrutura atual do projeto; não é necessário manifesto Silver prévio.


## 1 — Configuração e fontes Silver


In [1]:
from pathlib import Path
import sys
import duckdb
import pandas as pd
import time

RAIZ = Path.cwd()
if not (RAIZ / "src" / "enem_pipeline" / "config.py").exists():
    RAIZ = RAIZ.parent
assert (RAIZ / "src" / "enem_pipeline" / "config.py").exists(), "Não foi possível localizar src/enem_pipeline/config.py"
sys.path.insert(0, str(RAIZ / "src"))

from enem_pipeline.config import (
    PROJECT_ROOT, SILVER_DIR, GOLD_DIR, METADATA_DIR,
    UF_PROCESSAMENTO, ANO_INICIAL, ANO_FINAL,
)


UF_PROCESSAMENTO="RR"


GOLD_DIR = GOLD_DIR / f"uf={UF_PROCESSAMENTO}"
GOLD_DIR.mkdir(parents=True, exist_ok=True)
METADATA_DIR.mkdir(parents=True, exist_ok=True)
MANIFESTO_GOLD = METADATA_DIR / f"manifesto_gold_{UF_PROCESSAMENTO}.csv"

con = duckdb.connect()
fontes = []
for ano in range(ANO_INICIAL, ANO_FINAL + 1):
    diretorio = SILVER_DIR / f"uf={UF_PROCESSAMENTO}" / f"ano={ano}"
    bases = ([ ("enem", f"enem_{ano}.parquet") ] if ano <= 2023 else [
        ("participantes", "participantes.parquet"),
        ("resultados", "resultados.parquet"),
    ])
    for base, nome in bases:
        caminho = diretorio / nome
        if not caminho.is_file():
            raise FileNotFoundError(f"Fonte Silver ausente: {caminho}")
        fontes.append({"ano": ano, "base": base, "arquivo": str(caminho)})

fontes_silver = pd.DataFrame(fontes).sort_values(["ano", "base"]).reset_index(drop=True)
print(f"Raiz do projeto : {PROJECT_ROOT}")
print(f"UF              : {UF_PROCESSAMENTO}")
print(f"Fontes Silver   : {len(fontes_silver)}")
print(f"Edições         : {fontes_silver['ano'].nunique()}")
print(f"Diretório Gold  : {GOLD_DIR}")
display(fontes_silver)


Raiz do projeto : /home/akel/PycharmProjects/Projeto_ENEM
UF              : RR
Fontes Silver   : 30
Edições         : 28
Diretório Gold  : /home/akel/PycharmProjects/Projeto_ENEM/data/gold/uf=RR


,ano,base,arquivo
0,1998,enem,/home/akel/PycharmProjects/Projeto_ENEM/data/s...
1,1999,enem,/home/akel/PycharmProjects/Projeto_ENEM/data/s...
2,2000,enem,/home/akel/PycharmProjects/Projeto_ENEM/data/s...
3,2001,enem,/home/akel/PycharmProjects/Projeto_ENEM/data/s...
4,2002,enem,/home/akel/PycharmProjects/Projeto_ENEM/data/s...
5,2003,enem,/home/akel/PycharmProjects/Projeto_ENEM/data/s...
6,2004,enem,/home/akel/PycharmProjects/Projeto_ENEM/data/s...
7,2005,enem,/home/akel/PycharmProjects/Projeto_ENEM/data/s...
8,2006,enem,/home/akel/PycharmProjects/Projeto_ENEM/data/s...
9,2007,enem,/home/akel/PycharmProjects/Projeto_ENEM/data/s...


## 2 — Contrato de variáveis Gold


In [2]:

VARIAVEIS_GOLD = {
    "territorio": [
        "CO_MUNICIPIO_RESIDENCIA",
        "NO_MUNICIPIO_RESIDENCIA",
        "CO_UF_RESIDENCIA",
        "SG_UF_RESIDENCIA",

        "CO_MUNICIPIO_ESC",
        "NO_MUNICIPIO_ESC",
        "CO_UF_ESC",
        "SG_UF_ESC",

        "CO_MUNICIPIO_PROVA",
        "NO_MUNICIPIO_PROVA",
        "CO_UF_PROVA",
        "SG_UF_PROVA",
    ],

    "participante": [
        "TP_SEXO",
        "TP_FAIXA_ETARIA",
        "ANO_CONCLUIU",
    ],

    "presenca": [
        "TP_PRESENCA",
        "TP_PRESENCA_CN",
        "TP_PRESENCA_CH",
        "TP_PRESENCA_LC",
        "TP_PRESENCA_MT",
    ],

    "notas": [
        "NU_NOTA_CN",
        "NU_NOTA_CH",
        "NU_NOTA_LC",
        "NU_NOTA_MT",
        "NU_NOTA_OBJETIVA",
        "NU_NOTA_REDACAO",
    ],

    "redacao": [
        "TP_STATUS_REDACAO",
        "VL_PERC_COMP1",
        "VL_PERC_COMP2",
        "VL_PERC_COMP3",
        "VL_PERC_COMP4",
        "VL_PERC_COMP5",
    ],

    "escola": [
        "CO_ESCOLA",
        "TP_ESCOLA",
        "TP_DEPENDENCIA_ADM_ESC",
    ],
}

VARIAVEIS_GOLD_FLAT = [
    coluna
    for grupo in VARIAVEIS_GOLD.values()
    for coluna in grupo
]

print(f"Variáveis candidatas ao Gold: {len(VARIAVEIS_GOLD_FLAT)}")

Variáveis candidatas ao Gold: 35


## 3 — Contrato canônico de schema


In [3]:
COLUNAS_GOLD_CANONICAS = [
    "NU_ANO",

    "TIPO_REFERENCIA_TERRITORIAL",
    "SG_UF_TERRITORIO",
    "CO_UF_TERRITORIO",
    "CO_MUNICIPIO_TERRITORIO",
    "NO_MUNICIPIO_TERRITORIO",

    "TP_SEXO",
    "TP_FAIXA_ETARIA",
    "ANO_CONCLUIU",

    "TP_PRESENCA",
    "TP_PRESENCA_CN",
    "TP_PRESENCA_CH",
    "TP_PRESENCA_LC",
    "TP_PRESENCA_MT",

    "NU_NOTA_CN",
    "NU_NOTA_CH",
    "NU_NOTA_LC",
    "NU_NOTA_MT",
    "NU_NOTA_OBJETIVA",
    "NU_NOTA_REDACAO",

    "TP_STATUS_REDACAO",
    "VL_PERC_COMP1",
    "VL_PERC_COMP2",
    "VL_PERC_COMP3",
    "VL_PERC_COMP4",
    "VL_PERC_COMP5",

    "CO_ESCOLA",
    "TP_ESCOLA",
    "TP_DEPENDENCIA_ADM_ESC",
]

TIPOS_CANONICOS_GOLD = {
    "NU_ANO": "INTEGER",

    "TIPO_REFERENCIA_TERRITORIAL": "VARCHAR",
    "SG_UF_TERRITORIO": "VARCHAR",
    "CO_UF_TERRITORIO": "VARCHAR",
    "CO_MUNICIPIO_TERRITORIO": "VARCHAR",
    "NO_MUNICIPIO_TERRITORIO": "VARCHAR",

    "TP_SEXO": "VARCHAR",
    "TP_FAIXA_ETARIA": "INTEGER",
    "ANO_CONCLUIU": "INTEGER",

    "TP_PRESENCA": "INTEGER",
    "TP_PRESENCA_CN": "INTEGER",
    "TP_PRESENCA_CH": "INTEGER",
    "TP_PRESENCA_LC": "INTEGER",
    "TP_PRESENCA_MT": "INTEGER",

    "NU_NOTA_CN": "DOUBLE",
    "NU_NOTA_CH": "DOUBLE",
    "NU_NOTA_LC": "DOUBLE",
    "NU_NOTA_MT": "DOUBLE",
    "NU_NOTA_OBJETIVA": "DOUBLE",
    "NU_NOTA_REDACAO": "DOUBLE",

    "TP_STATUS_REDACAO": "VARCHAR",
    "VL_PERC_COMP1": "DOUBLE",
    "VL_PERC_COMP2": "DOUBLE",
    "VL_PERC_COMP3": "DOUBLE",
    "VL_PERC_COMP4": "DOUBLE",
    "VL_PERC_COMP5": "DOUBLE",

    "CO_ESCOLA": "VARCHAR",
    "TP_ESCOLA": "INTEGER",
    "TP_DEPENDENCIA_ADM_ESC": "INTEGER",
}

assert set(COLUNAS_GOLD_CANONICAS) == set(TIPOS_CANONICOS_GOLD)

schema_canonico_gold = pd.DataFrame({
    "ordem": range(1, len(COLUNAS_GOLD_CANONICAS) + 1),
    "coluna": COLUNAS_GOLD_CANONICAS,
    "tipo_canonico": [
        TIPOS_CANONICOS_GOLD[coluna]
        for coluna in COLUNAS_GOLD_CANONICAS
    ]
})

display(schema_canonico_gold)

print(f"\n✓ Colunas canônicas: {len(COLUNAS_GOLD_CANONICAS)}")
print(f"✓ Tipos definidos: {len(TIPOS_CANONICOS_GOLD)}")
print("✓ Contrato completo: True")


,ordem,coluna,tipo_canonico
0,1,NU_ANO,INTEGER
1,2,TIPO_REFERENCIA_TERRITORIAL,VARCHAR
2,3,SG_UF_TERRITORIO,VARCHAR
3,4,CO_UF_TERRITORIO,VARCHAR
4,5,CO_MUNICIPIO_TERRITORIO,VARCHAR
5,6,NO_MUNICIPIO_TERRITORIO,VARCHAR
6,7,TP_SEXO,VARCHAR
7,8,TP_FAIXA_ETARIA,INTEGER
8,9,ANO_CONCLUIU,INTEGER
9,10,TP_PRESENCA,INTEGER



✓ Colunas canônicas: 29
✓ Tipos definidos: 29
✓ Contrato completo: True


## 4 — Referência territorial


In [4]:
def definir_territorio(ano):
    if 1998 <= ano <= 2000:
        return {
            "tipo": "RESIDENCIA",
            "codigo": "CO_MUNICIPIO_RESIDENCIA",
            "nome": "NO_MUNICIPIO_RESIDENCIA",
            "uf_codigo": "CO_UF_RESIDENCIA",
            "uf_sigla": "SG_UF_RESIDENCIA",
        }

    if 2001 <= ano <= 2007:
        return {
            "tipo": "ESC",
            "codigo": "CO_MUNICIPIO_ESC",
            "nome": "NO_MUNICIPIO_ESC",
            "uf_codigo": "CO_UF_ESC",
            "uf_sigla": "SG_UF_ESC",
        }

    if 2008 <= ano <= 2025:
        return {
            "tipo": "PROVA",
            "codigo": "CO_MUNICIPIO_PROVA",
            "nome": "NO_MUNICIPIO_PROVA",
            "uf_codigo": "CO_UF_PROVA",
            "uf_sigla": "SG_UF_PROVA",
        }

    raise ValueError(f"Ano fora do escopo: {ano}")


## 5 — Construção do SELECT canônico


In [5]:
def construir_select_gold(ano, base, arquivo):
    """Constrói o SELECT canônico da camada Gold."""

    territorio = definir_territorio(ano)

    colunas_fonte = set(
        con.sql(
            f"""
            DESCRIBE
            SELECT *
            FROM read_parquet(
                '{arquivo}',
                hive_partitioning = false
            )
            """
        ).df()["column_name"]
    )

    def campo(nome, alias=None):
        destino = alias or nome
        tipo = TIPOS_CANONICOS_GOLD[destino]

        if nome in colunas_fonte:
            return (
                f'CAST("{nome}" AS {tipo}) '
                f'AS "{destino}"'
            )

        return (
            f'CAST(NULL AS {tipo}) '
            f'AS "{destino}"'
        )

    select = [
        campo("NU_ANO"),

        (
            f"CAST('{territorio['tipo']}' AS VARCHAR) "
            f'AS "TIPO_REFERENCIA_TERRITORIAL"'
        ),

        campo(territorio["uf_sigla"], "SG_UF_TERRITORIO"),
        campo(territorio["uf_codigo"], "CO_UF_TERRITORIO"),
        campo(territorio["codigo"], "CO_MUNICIPIO_TERRITORIO"),
        campo(territorio["nome"], "NO_MUNICIPIO_TERRITORIO"),

        campo("TP_SEXO"),
        campo("TP_FAIXA_ETARIA"),
        campo("ANO_CONCLUIU"),

        campo("TP_PRESENCA"),
        campo("TP_PRESENCA_CN"),
        campo("TP_PRESENCA_CH"),
        campo("TP_PRESENCA_LC"),
        campo("TP_PRESENCA_MT"),

        campo("NU_NOTA_CN"),
        campo("NU_NOTA_CH"),
        campo("NU_NOTA_LC"),
        campo("NU_NOTA_MT"),
        campo("NU_NOTA_OBJETIVA"),
        campo("NU_NOTA_REDACAO"),

        campo("TP_STATUS_REDACAO"),
        campo("VL_PERC_COMP1"),
        campo("VL_PERC_COMP2"),
        campo("VL_PERC_COMP3"),
        campo("VL_PERC_COMP4"),
        campo("VL_PERC_COMP5"),

        campo("CO_ESCOLA"),
        campo("TP_ESCOLA"),
        campo("TP_DEPENDENCIA_ADM_ESC"),
    ]

    return ",\n        ".join(select)


## 6 — Materialização do Gold físico


In [6]:
total_fontes = len(fontes_silver)

for i, (_, fonte) in enumerate(
    fontes_silver.iterrows(),
    start=1
):

    ano = int(fonte["ano"])
    base = fonte["base"]
    arquivo = fonte["arquivo"]

    print(
        f"\n[{i:02d}/{total_fontes:02d}] "
        f"Processando {ano} | {base}"
    )

    select_gold = construir_select_gold(
        ano=ano,
        base=base,
        arquivo=arquivo
    )

    destino = (
        GOLD_DIR
        / f"ano={ano}"
        / f"base={base}"
    )

    destino.mkdir(
        parents=True,
        exist_ok=True
    )

    arquivo_saida = destino / "dados.parquet"
    arquivo_temporario = destino / "dados_gold_tmp.parquet"

    if arquivo_temporario.exists():
        arquivo_temporario.unlink()

    con.sql(
        f"""
        COPY (
            SELECT
                {select_gold}
            FROM read_parquet(
                '{arquivo}',
                hive_partitioning = false
            )
        )
        TO '{arquivo_temporario}'
        (FORMAT PARQUET, COMPRESSION ZSTD)
        """
    )

    schema_tmp = con.sql(
        f"""
        DESCRIBE
        SELECT *
        FROM read_parquet(
            '{arquivo_temporario}',
            hive_partitioning = false
        )
        """
    ).df()

    schema_valido = (
        schema_tmp["column_name"].tolist()
        == COLUNAS_GOLD_CANONICAS
        and
        schema_tmp["column_type"]
        .str.upper()
        .tolist()
        ==
        [
            TIPOS_CANONICOS_GOLD[coluna]
            for coluna in COLUNAS_GOLD_CANONICAS
        ]
    )

    if not schema_valido:
        arquivo_temporario.unlink()
        raise ValueError(
            f"Schema inválido para {ano} | {base}"
        )

    arquivo_temporario.replace(arquivo_saida)

    print(
        f"✓ {ano} | {base} → {arquivo_saida}"
    )

print("\n" + "=" * 70)
print("RECONSTRUÇÃO DA GOLD: CONCLUÍDA")
print("=" * 70)
print(f"✓ Fontes reconstruídas: {total_fontes}")



[01/30] Processando 1998 | enem
✓ 1998 | enem → /home/akel/PycharmProjects/Projeto_ENEM/data/gold/uf=RR/ano=1998/base=enem/dados.parquet

[02/30] Processando 1999 | enem
✓ 1999 | enem → /home/akel/PycharmProjects/Projeto_ENEM/data/gold/uf=RR/ano=1999/base=enem/dados.parquet

[03/30] Processando 2000 | enem
✓ 2000 | enem → /home/akel/PycharmProjects/Projeto_ENEM/data/gold/uf=RR/ano=2000/base=enem/dados.parquet

[04/30] Processando 2001 | enem
✓ 2001 | enem → /home/akel/PycharmProjects/Projeto_ENEM/data/gold/uf=RR/ano=2001/base=enem/dados.parquet

[05/30] Processando 2002 | enem
✓ 2002 | enem → /home/akel/PycharmProjects/Projeto_ENEM/data/gold/uf=RR/ano=2002/base=enem/dados.parquet

[06/30] Processando 2003 | enem
✓ 2003 | enem → /home/akel/PycharmProjects/Projeto_ENEM/data/gold/uf=RR/ano=2003/base=enem/dados.parquet

[07/30] Processando 2004 | enem
✓ 2004 | enem → /home/akel/PycharmProjects/Projeto_ENEM/data/gold/uf=RR/ano=2004/base=enem/dados.parquet

[08/30] Processando 2005 | enem
✓

## 7 — Compatibilidade municipal de 2024–2025

A compatibilidade entre `PARTICIPANTES` e `RESULTADOS` é verificada por município. Não é realizado join individual.


In [7]:

for ano in [2024, 2025]:

    participantes = (
        GOLD_DIR
        / f"ano={ano}"
        / "base=participantes"
        / "dados.parquet"
    )

    resultados = (
        GOLD_DIR
        / f"ano={ano}"
        / "base=resultados"
        / "dados.parquet"
    )

    comparacao = con.sql(
        f"""
        WITH participantes AS (
            SELECT
                CO_MUNICIPIO_TERRITORIO,
                NO_MUNICIPIO_TERRITORIO,
                COUNT(*) AS n_participantes
            FROM read_parquet('{participantes}')
            GROUP BY 1, 2
        ),

        resultados AS (
            SELECT
                CO_MUNICIPIO_TERRITORIO,
                NO_MUNICIPIO_TERRITORIO,
                COUNT(*) AS n_resultados
            FROM read_parquet('{resultados}')
            GROUP BY 1, 2
        )

        SELECT
            COALESCE(p.CO_MUNICIPIO_TERRITORIO,
                     r.CO_MUNICIPIO_TERRITORIO)
                AS CO_MUNICIPIO_TERRITORIO,

            COALESCE(p.NO_MUNICIPIO_TERRITORIO,
                     r.NO_MUNICIPIO_TERRITORIO)
                AS NO_MUNICIPIO_TERRITORIO,

            COALESCE(p.n_participantes, 0) AS n_participantes,
            COALESCE(r.n_resultados, 0) AS n_resultados,

            COALESCE(p.n_participantes, 0)
                - COALESCE(r.n_resultados, 0) AS diferenca

        FROM participantes p
        FULL OUTER JOIN resultados r
            ON p.CO_MUNICIPIO_TERRITORIO =
               r.CO_MUNICIPIO_TERRITORIO

        ORDER BY CO_MUNICIPIO_TERRITORIO
        """
    ).df()

    print(f"\n{'=' * 70}")
    print(f"ANO {ano}")
    print(f"{'=' * 70}")

    print(f"Municípios participantes: {len(comparacao[comparacao.n_participantes > 0])}")

    print(f"Municípios resultados: {len(comparacao[comparacao.n_resultados > 0])}")

    print(f"Municípios com contagens iguais: {(comparacao['diferenca'] == 0).sum()}")

    print(f"Municípios com diferença: {(comparacao['diferenca'] != 0).sum()}")

    display(comparacao[comparacao["diferenca"] != 0])


ANO 2024
Municípios participantes: 7
Municípios resultados: 7
Municípios com contagens iguais: 7
Municípios com diferença: 0


,CO_MUNICIPIO_TERRITORIO,NO_MUNICIPIO_TERRITORIO,n_participantes,n_resultados,diferenca



ANO 2025
Municípios participantes: 10
Municípios resultados: 10
Municípios com contagens iguais: 10
Municípios com diferença: 0


,CO_MUNICIPIO_TERRITORIO,NO_MUNICIPIO_TERRITORIO,n_participantes,n_resultados,diferenca


## 8 — Manifesto Gold


In [8]:

manifesto_gold = fontes_silver[
    ["ano", "base", "arquivo"]
].copy()

manifesto_gold["camada"] = "gold"
manifesto_gold["uf"] = UF_PROCESSAMENTO
manifesto_gold["tipo_referencia_territorial"] = (
    manifesto_gold["ano"]
    .apply(lambda x: definir_territorio(int(x))["tipo"])
)
manifesto_gold["unidade_integracao"] = "ano_municipio"
manifesto_gold["join_individual"] = False
manifesto_gold["arquivo_gold"] = manifesto_gold.apply(
    lambda r: str(
        GOLD_DIR
        / f"ano={int(r['ano'])}"
        / f"base={r['base']}"
        / "dados.parquet"
    ),
    axis=1
)

manifesto_gold = manifesto_gold[
    [
        "camada",
        "uf",
        "ano",
        "base",
        "tipo_referencia_territorial",
        "unidade_integracao",
        "join_individual",
        "arquivo",
        "arquivo_gold",
    ]
].sort_values(["ano", "base"])

display(manifesto_gold)

,camada,uf,ano,base,tipo_referencia_territorial,unidade_integracao,join_individual,arquivo,arquivo_gold
0,gold,RR,1998,enem,RESIDENCIA,ano_municipio,False,/home/akel/PycharmProjects/Projeto_ENEM/data/s...,/home/akel/PycharmProjects/Projeto_ENEM/data/g...
1,gold,RR,1999,enem,RESIDENCIA,ano_municipio,False,/home/akel/PycharmProjects/Projeto_ENEM/data/s...,/home/akel/PycharmProjects/Projeto_ENEM/data/g...
2,gold,RR,2000,enem,RESIDENCIA,ano_municipio,False,/home/akel/PycharmProjects/Projeto_ENEM/data/s...,/home/akel/PycharmProjects/Projeto_ENEM/data/g...
3,gold,RR,2001,enem,ESC,ano_municipio,False,/home/akel/PycharmProjects/Projeto_ENEM/data/s...,/home/akel/PycharmProjects/Projeto_ENEM/data/g...
4,gold,RR,2002,enem,ESC,ano_municipio,False,/home/akel/PycharmProjects/Projeto_ENEM/data/s...,/home/akel/PycharmProjects/Projeto_ENEM/data/g...
5,gold,RR,2003,enem,ESC,ano_municipio,False,/home/akel/PycharmProjects/Projeto_ENEM/data/s...,/home/akel/PycharmProjects/Projeto_ENEM/data/g...
6,gold,RR,2004,enem,ESC,ano_municipio,False,/home/akel/PycharmProjects/Projeto_ENEM/data/s...,/home/akel/PycharmProjects/Projeto_ENEM/data/g...
7,gold,RR,2005,enem,ESC,ano_municipio,False,/home/akel/PycharmProjects/Projeto_ENEM/data/s...,/home/akel/PycharmProjects/Projeto_ENEM/data/g...
8,gold,RR,2006,enem,ESC,ano_municipio,False,/home/akel/PycharmProjects/Projeto_ENEM/data/s...,/home/akel/PycharmProjects/Projeto_ENEM/data/g...
9,gold,RR,2007,enem,ESC,ano_municipio,False,/home/akel/PycharmProjects/Projeto_ENEM/data/s...,/home/akel/PycharmProjects/Projeto_ENEM/data/g...


In [9]:

manifesto_gold.to_csv(
    MANIFESTO_GOLD,
    sep=";",
    index=False,
    encoding="utf-8"
)

print(f"✓ Manifesto Gold salvo:")
print(MANIFESTO_GOLD)
print(f"✓ Fontes registradas: {len(manifesto_gold)}")

✓ Manifesto Gold salvo:
/home/akel/PycharmProjects/Projeto_ENEM/data/metadata/manifesto_gold_RR.csv
✓ Fontes registradas: 30


## 9 — Verificação final do schema Gold

A auditoria completa permanece no notebook `09_auditoria_gold.ipynb`. Aqui fazemos apenas uma verificação final de que as 30 fontes materializadas respeitam o contrato canônico.


In [10]:
print("=" * 70)
print("VERIFICAÇÃO FINAL — SCHEMA GOLD")
print("=" * 70)

falhas = []

for _, fonte in manifesto_gold.iterrows():

    ano = int(fonte["ano"])
    base = fonte["base"]
    arquivo_gold = fonte["arquivo_gold"]

    schema = con.sql(
        f"""
        DESCRIBE
        SELECT *
        FROM read_parquet(
            '{arquivo_gold}',
            hive_partitioning = false
        )
        """
    ).df()

    nomes_ok = (
        schema["column_name"].tolist()
        == COLUNAS_GOLD_CANONICAS
    )

    tipos_ok = (
        schema["column_type"]
        .str.upper()
        .tolist()
        ==
        [
            TIPOS_CANONICOS_GOLD[coluna]
            for coluna in COLUNAS_GOLD_CANONICAS
        ]
    )

    if not (nomes_ok and tipos_ok):
        falhas.append(f"{ano} | {base}")

print(
    f"✓ Fontes verificadas: {len(manifesto_gold)}/{len(manifesto_gold)}"
)
print(
    f"✓ Fontes com schema válido: {len(manifesto_gold) - len(falhas)}/{len(manifesto_gold)}"
)

if falhas:
    print("\nFontes com falha:")
    for falha in falhas:
        print(f"  - {falha}")
    raise ValueError("Schema Gold inválido em uma ou mais fontes.")

print("\n✓ SCHEMA GOLD: OK")
print(f"\n# Processo finalizado em: {time.strftime('%d/%m/%Y %H:%M:%S')}")

VERIFICAÇÃO FINAL — SCHEMA GOLD
✓ Fontes verificadas: 30/30
✓ Fontes com schema válido: 30/30

✓ SCHEMA GOLD: OK

# Processo finalizado em: 17/10/2026 14:58:54


## validar notas e códigos municipais na Gold

In [11]:
from pathlib import Path
import duckdb

con = duckdb.connect()

gold_dir = GOLD_DIR # / f"uf={UF_PROCESSAMENTO}"

arquivos = sorted(gold_dir.glob("ano=*/base=*/dados.parquet"))

consulta = """
SELECT
    filename,
    COUNT(*) AS linhas,
    COUNT(CO_MUNICIPIO_TERRITORIO) AS com_codigo_municipio,
    COUNT(NU_NOTA_REDACAO) AS com_nota_redacao,
    COUNT(NU_NOTA_CN) AS com_nota_cn,
    COUNT(NU_NOTA_CH) AS com_nota_ch,
    COUNT(NU_NOTA_LC) AS com_nota_lc,
    COUNT(NU_NOTA_MT) AS com_nota_mt
FROM read_parquet(?, union_by_name=True, filename=True)
GROUP BY filename
ORDER BY filename
"""

if not arquivos:
    raise FileNotFoundError(f"Nenhum Parquet encontrado em {gold_dir}")

df_validacao = con.execute(
    consulta,
    [[str(p) for p in arquivos]]
).df()

display(df_validacao)

,filename,linhas,com_codigo_municipio,com_nota_redacao,com_nota_cn,com_nota_ch,com_nota_lc,com_nota_mt
0,/home/akel/PycharmProjects/Projeto_ENEM/data/g...,967,967,967,0,0,0,0
1,/home/akel/PycharmProjects/Projeto_ENEM/data/g...,1014,1014,1014,0,0,0,0
2,/home/akel/PycharmProjects/Projeto_ENEM/data/g...,317,317,317,0,0,0,0
3,/home/akel/PycharmProjects/Projeto_ENEM/data/g...,2606,2606,2606,0,0,0,0
4,/home/akel/PycharmProjects/Projeto_ENEM/data/g...,5497,5497,5497,0,0,0,0
5,/home/akel/PycharmProjects/Projeto_ENEM/data/g...,6316,6316,6316,0,0,0,0
6,/home/akel/PycharmProjects/Projeto_ENEM/data/g...,4793,4793,4793,0,0,0,0
7,/home/akel/PycharmProjects/Projeto_ENEM/data/g...,5986,5986,5986,0,0,0,0
8,/home/akel/PycharmProjects/Projeto_ENEM/data/g...,2810,2810,2810,0,0,0,0
9,/home/akel/PycharmProjects/Projeto_ENEM/data/g...,4207,4207,4207,0,0,0,0


In [13]:

from pathlib import Path
import re
import duckdb
import pandas as pd

# ============================================================
# AUDITORIA COMPLETA — GOLD
# ============================================================

gold_dir = Path(GOLD_DIR)
con = duckdb.connect()

arquivos = sorted(gold_dir.glob("ano=*/base=*/dados.parquet"))

if not arquivos:
    raise FileNotFoundError(f"Nenhum Parquet encontrado em {gold_dir}")

colunas_obrigatorias = [
    "NU_ANO",
    "TIPO_REFERENCIA_TERRITORIAL",
    "SG_UF_TERRITORIO",
    "CO_UF_TERRITORIO",
    "CO_MUNICIPIO_TERRITORIO",
    "NO_MUNICIPIO_TERRITORIO",
]

notas = [
    "NU_NOTA_REDACAO",
    "NU_NOTA_CN",
    "NU_NOTA_CH",
    "NU_NOTA_LC",
    "NU_NOTA_MT",
    "NU_NOTA_OBJETIVA",
]

# ------------------------------------------------------------
# 1. Estrutura, schema e integridade por arquivo
# ------------------------------------------------------------

auditoria = []
schemas = {}
erros = []

for arquivo in arquivos:
    partes = dict(re.findall(r"(uf|ano|base)=([^/]+)", str(arquivo)))
    ano_particao = int(partes["ano"])
    base = partes["base"]

    try:
        descricao = con.execute(
            "DESCRIBE SELECT * FROM read_parquet(?)",
            [str(arquivo)]
        ).fetchall()

        schema = [(r[0], r[1]) for r in descricao]
        schemas[str(arquivo)] = schema
        colunas = {nome for nome, _ in schema}

        faltantes = sorted(set(colunas_obrigatorias) - colunas)

        # Verificações básicas
        expressoes = [
            "COUNT(*) AS linhas",
            "COUNT(DISTINCT CO_MUNICIPIO_TERRITORIO) "
            "AS municipios",
            "COUNT(*) FILTER "
            "(WHERE CO_MUNICIPIO_TERRITORIO IS NULL) "
            "AS municipio_codigo_nulo",
            "COUNT(*) FILTER "
            "(WHERE NO_MUNICIPIO_TERRITORIO IS NULL) "
            "AS municipio_nome_nulo",
            f"COUNT(*) FILTER (WHERE NU_ANO IS NULL "
            f"OR TRY_CAST(NU_ANO AS INTEGER) != {ano_particao}) "
            "AS ano_inconsistente",
        ]

        for coluna in notas:
            if coluna in colunas:
                expressoes.append(
                    f'COUNT("{coluna}") AS "{coluna}_preenchida"'
                )
                expressoes.append(
                    f'COUNT(*) FILTER (WHERE "{coluna}" < 0 '
                    f'OR "{coluna}" > 1000) AS "{coluna}_fora_faixa"'
                )

        for coluna in [
            "CO_MUNICIPIO_TERRITORIO",
            "CO_UF_TERRITORIO",
        ]:
            if coluna in colunas:
                expressoes.append(
                    f'COUNT(*) FILTER (WHERE '
                    f'TRY_CAST("{coluna}" AS BIGINT) IS NULL '
                    f'AND "{coluna}" IS NOT NULL) '
                    f'AS "{coluna}_codigo_invalido"'
                )

        resultado = con.execute(
            f"SELECT {', '.join(expressoes)} "
            "FROM read_parquet(?)",
            [str(arquivo)]
        ).df().iloc[0].to_dict()

        auditoria.append({
            "arquivo": str(arquivo),
            "ano": ano_particao,
            "base": base,
            "schema_colunas": len(schema),
            "colunas_obrigatorias_faltantes": ", ".join(faltantes),
            **resultado,
        })

    except Exception as exc:
        erros.append({
            "arquivo": str(arquivo),
            "erro": str(exc),
        })

df_auditoria = pd.DataFrame(auditoria)

# ------------------------------------------------------------
# 2. Consistência do schema entre os arquivos
# ------------------------------------------------------------

schema_base = next(iter(schemas.values()), None)

df_schema = pd.DataFrame([
    {
        "arquivo": caminho,
        "schema_compativel": schema == schema_base,
        "colunas": len(schema),
    }
    for caminho, schema in schemas.items()
])

# ------------------------------------------------------------
# 3. Comparação de linhas Gold versus origem Silver
#    Usa os caminhos registrados no manifesto, se disponível.
# ------------------------------------------------------------

manifestos = list(gold_dir.glob("manifesto_gold_*.csv"))
if not manifestos:
    manifestos = list(gold_dir.parent.glob("manifesto_gold_*.csv"))

comparacoes = []

for manifesto_path in manifestos:
    try:
        manifesto = pd.read_csv(manifesto_path, sep=";")

        for _, linha in manifesto.iterrows():
            origem = Path(str(linha["arquivo"]))
            destino = Path(str(linha["arquivo_gold"]))

            item = {
                "ano": linha["ano"],
                "base": linha["base"],
                "origem_existe": origem.is_file(),
                "gold_existe": destino.is_file(),
                "linhas_origem": None,
                "linhas_gold": None,
                "contagens_iguais": False,
            }

            try:
                if origem.is_file():
                    item["linhas_origem"] = con.execute(
                        "SELECT COUNT(*) FROM read_parquet(?)",
                        [str(origem)]
                    ).fetchone()[0]

                if destino.is_file():
                    item["linhas_gold"] = con.execute(
                        "SELECT COUNT(*) FROM read_parquet(?)",
                        [str(destino)]
                    ).fetchone()[0]

                item["contagens_iguais"] = (
                    item["linhas_origem"] is not None
                    and item["linhas_gold"] is not None
                    and item["linhas_origem"] == item["linhas_gold"]
                )

            except Exception as exc:
                item["erro"] = str(exc)

            comparacoes.append(item)

    except Exception as exc:
        erros.append({
            "arquivo": str(manifesto_path),
            "erro": f"Falha ao ler manifesto: {exc}",
        })

df_comparacao = pd.DataFrame(comparacoes)

# ------------------------------------------------------------
# 4. Resumo e relatório de problemas
# ------------------------------------------------------------

print("=" * 65)
print("AUDITORIA GOLD")
print("=" * 65)
print(f"Arquivos encontrados: {len(arquivos)}")
print(f"Arquivos auditados: {len(df_auditoria)}")
print(f"Erros de leitura: {len(erros)}")
print(f"Schemas divergentes: "
      f"{int((~df_schema['schema_compativel']).sum()) if not df_schema.empty else 0}")

if not df_auditoria.empty:
    print("\nRESUMO POR ANO E BASE")
    colunas_resumo = [
        c for c in [
            "ano", "base", "linhas", "municipios",
            "municipio_codigo_nulo", "municipio_nome_nulo",
            "ano_inconsistente",
        ]
        if c in df_auditoria.columns
    ]
    display(df_auditoria[colunas_resumo])

    print("\nARQUIVOS COM COLUNAS OBRIGATÓRIAS AUSENTES")
    display(df_auditoria[
        df_auditoria["colunas_obrigatorias_faltantes"] != ""
    ][["arquivo", "colunas_obrigatorias_faltantes"]])

    colunas_faixa = [
        c for c in df_auditoria.columns if c.endswith("_fora_faixa")
    ]
    if colunas_faixa:
        print("\nNOTAS FORA DA FAIXA 0–1000")
        display(df_auditoria[
            (df_auditoria[colunas_faixa] > 0).any(axis=1)
        ][["arquivo"] + colunas_faixa])

if not df_comparacao.empty:
    print("\nCONTAGEM DE LINHAS: SILVER × GOLD")
    display(df_comparacao)

if erros:
    print("\nERROS")
    display(pd.DataFrame(erros))

print("\nAUDITORIA CONCLUÍDA — nenhum arquivo foi alterado.")


AUDITORIA GOLD
Arquivos encontrados: 30
Arquivos auditados: 30
Erros de leitura: 0
Schemas divergentes: 0

RESUMO POR ANO E BASE


,ano,base,linhas,municipios,municipio_codigo_nulo,municipio_nome_nulo,ano_inconsistente
0,1998,enem,967,4,0,0,0
1,1999,enem,1014,10,0,0,0
2,2000,enem,317,9,0,0,0
3,2001,enem,2606,14,0,0,0
4,2002,enem,5497,15,0,0,0
5,2003,enem,6316,15,0,0,0
6,2004,enem,4793,15,0,0,0
7,2005,enem,5986,15,0,0,0
8,2006,enem,2810,14,0,0,0
9,2007,enem,4207,15,0,0,0



ARQUIVOS COM COLUNAS OBRIGATÓRIAS AUSENTES


,arquivo,colunas_obrigatorias_faltantes



NOTAS FORA DA FAIXA 0–1000


,arquivo,NU_NOTA_REDACAO_fora_faixa,NU_NOTA_CN_fora_faixa,NU_NOTA_CH_fora_faixa,NU_NOTA_LC_fora_faixa,NU_NOTA_MT_fora_faixa,NU_NOTA_OBJETIVA_fora_faixa



AUDITORIA CONCLUÍDA — nenhum arquivo foi alterado.


In [18]:
from pathlib import Path
import duckdb
import pandas as pd

con = duckdb.connect()

# Localizar manifesto
raiz = Path(GOLD_DIR)
manifestos = list(Path(METADATA_DIR).glob("manifesto_gold_*.csv"))
if not manifestos:
    manifestos = list(raiz.glob("manifesto_gold_*.csv"))

if not manifestos:
    raise FileNotFoundError("Manifesto Gold não encontrado.")

manifesto = pd.read_csv(manifestos[0], sep=";")
resultados = []

for _, row in manifesto.iterrows():
    origem = Path(row["arquivo"])
    destino = Path(row["arquivo_gold"])

    item = {
        "ano": row["ano"],
        "base": row["base"],
        "origem_existe": origem.exists(),
        "gold_existe": destino.exists(),
    }

    if origem.exists() and destino.exists():
        item["linhas_silver"] = con.execute(
            "SELECT COUNT(*) FROM read_parquet(?)", [str(origem)]
        ).fetchone()[0]
        item["linhas_gold"] = con.execute(
            "SELECT COUNT(*) FROM read_parquet(?)", [str(destino)]
        ).fetchone()[0]
        item["contagens_iguais"] = (
            item["linhas_silver"] == item["linhas_gold"]
        )

        item["duplicatas_gold"] = con.execute(
            """
            SELECT COUNT(*) FROM (
                SELECT * FROM read_parquet(?)
                GROUP BY ALL
                HAVING COUNT(*) > 1
            )
            """,
            [str(destino)],
        ).fetchone()[0]

    resultados.append(item)

df_final = pd.DataFrame(resultados)
display(df_final)

print("Contagens Silver × Gold iguais:",
      int(df_final["contagens_iguais"].sum()),
      "/", len(df_final))
print("Arquivos ausentes:",
      int((~df_final["origem_existe"] | ~df_final["gold_existe"]).sum()))

,ano,base,origem_existe,gold_existe,linhas_silver,linhas_gold,contagens_iguais,duplicatas_gold
0,1998,enem,True,True,967,967,True,14
1,1999,enem,True,True,1014,1014,True,26
2,2000,enem,True,True,317,317,True,15
3,2001,enem,True,True,2606,2606,True,64
4,2002,enem,True,True,5497,5497,True,158
5,2003,enem,True,True,6316,6316,True,225
6,2004,enem,True,True,4793,4793,True,187
7,2005,enem,True,True,5986,5986,True,175
8,2006,enem,True,True,2810,2810,True,84
9,2007,enem,True,True,4207,4207,True,119


Contagens Silver × Gold iguais: 30 / 30
Arquivos ausentes: 0


In [16]:
manifestos

[]

In [17]:
raiz

PosixPath('/home/akel/PycharmProjects/Projeto_ENEM/data/gold/uf=RR')